In [22]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re

In [23]:
HEADERS = {
    "User-Agent": "Mozilla/5.0"
}


In [24]:
def load_source_urls(file_path):
    urls = []
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()

            if not line:
                continue

            # remove bullet
            if line.startswith("-"):
                line = line.replace("- ", "")

            # remove arrow mapping
            if "→" in line:
                line = line.split("→")[0].strip()

            if line.startswith("http"):
                urls.append(line)

    return urls

In [25]:
def scrape_metric_page(url):
    response = requests.get(url, headers=HEADERS, timeout=30)

    if response.status_code != 200:
        return {}

    soup = BeautifulSoup(response.text, "html.parser")

    # Robust selector (site-safe)
    records = soup.select("div.recordsetContainer")

    data = {}

    for rec in records:
        try:
            country = rec.select_one("span.textShadow").text.strip()
            value = rec.select("span.textLarge")[-1].text.strip()
            data[country] = value
        except:
            continue

    return data

In [26]:
def build_military_dataset(urls):
    final_df = pd.DataFrame()

    for url in urls:
        metric_name = url.split("/")[-1].replace(".php", "")
        metric_data = scrape_metric_page(url)

        # skip empty scrapes
        if not metric_data:
            continue

        temp_df = pd.DataFrame(
            list(metric_data.items()),
            columns=["country", metric_name]
        )

        if final_df.empty:
            final_df = temp_df
        else:
            final_df = final_df.merge(
                temp_df, on="country", how="outer"
            )

    return final_df

In [27]:
def clean_numeric_columns(df):
    for col in df.columns:
        if col == "country":
            continue

        cleaned = []
        for val in df[col]:
            if pd.isna(val):
                cleaned.append(None)
                continue

            val = str(val).replace(",", "").replace(" ", "")
            match = re.search(r"-?\d+\.?\d*", val)

            cleaned.append(float(match.group()) if match else None)

        df[col] = cleaned

    return df

In [28]:
urls = load_source_urls("links_for_military_data.txt")
print("URLs loaded:", len(urls))

URLs loaded: 54


In [29]:
df = build_military_dataset(urls)
df = clean_numeric_columns(df)

In [30]:
print("Final shape:", df.shape)
print(df.head())

Final shape: (145, 55)
       country  total-population-by-country  available-military-manpower  \
0  Afghanistan                   40121552.0                   15647405.0   
1      Albania                    3107100.0                    1522479.0   
2      Algeria                   47022473.0                   22570787.0   
3       Angola                   37202061.0                    7440412.0   
4    Argentina                   46994384.0                   20677529.0   

   manpower-fit-for-military-service  manpower-reaching-military-age-annually  \
0                          8826741.0                                 842553.0   
1                          1292554.0                                  62142.0   
2                         19185169.0                                 752360.0   
3                          3720206.0                                 372021.0   
4                         17575900.0                                 704916.0   

   active-military-manpower  acti

In [31]:
df.to_csv("military_raw_data.csv", index=False)

In [32]:
print("CSV saved successfully.")

CSV saved successfully.
